In [ ]:
# pip install tensorflow opencv-python numpy pandas matplotlib seaborn scikit-learn streamlit pillow

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/anourhamuze/neurovisiondata"):
    print(f"\nDossier : {root}")
    print(f"Sous-dossiers : {dirs}")
    print(f"Nombre de fichiers : {len(files)}")

In [ ]:
from pathlib import Path
data_path = Path("/kaggle/input/datasets/anourhamuze/neurovisiondata/Data")
extensions = {".jpg", ".jpeg", ".png", ".bmp"}
# classes = np.array(sorted([item.name for item in data_path.glob("*")]))
# print(classes)
classes = sorted(dossier.name for dossier in data_path.iterdir()if dossier.is_dir())
print(classes)
image_diff_ex=0
resultats = []
etiquette = []
images = []
for classe in classes:
    dossier = data_path/classe
    for f in dossier.rglob("*"):
      if f.is_file():
        if f.suffix.lower() in extensions:
            images.append(f)
            etiquette.append(classe)
        else:
            image_diff_ex += 1
    resultats.append({"Classe": classe, "Nombre d'images": len(images)})
print(resultats)
# print(image_diff_ex)

In [ ]:
images_invalides = []
for img in images:
    try:
        image = cv2.imread(str(img))
        if image is None:
            images_invalides.append(str(img))
            continue
    except Exception as e:
        images_invalides.append(str(img))
        print(f"Erreur pour {f} : {e}")
print(images_invalides)

In [ ]:
# import random
# combined = list(zip(images, etiquette))
# random.seed(42)
# random.shuffle(combined)
# images, labels = zip(*donnees)
# images = list(images)
# labels = list(labels)
# print(combined)

In [ ]:

# images_redimensionnees = []
# etiquettes_valides = []

# for chemin, label in zip(images, etiquette):
#     image = cv2.imread(str(chemin))

#     if image is not None:
#         image_resize = cv2.resize(image, (224, 224))
#         images_redimensionnees.append(image_resize)
#         etiquettes_valides.append(label)

# images = np.array(images_redimensionnees)
# etiquette = np.array(etiquettes_valides)



In [ ]:
# images_redimensionnees = []
# for img in images:
#     image = cv2.imread(str(img))
#     image_resize = cv2.resize(image, (224, 224))
#     images_redimensionnees.append(image_resize)
# images = np.array(images_redimensionnees)
# etiquette = np.array(etiquette)
# print(etiquette)


In [ ]:
def charger_images(images):
    images_charge = []
    etiquettes = []
    for img, label in zip(images, etiquette):
        image = cv2.imread(str(img))
        image = cv2.resize(image, (224, 224))
        images_charge.append(image)
        etiquettes.append(label)
    return images_charge, etiquettes

In [ ]:
images, etiquette = charger_images(images)


import random

combined = list(zip(images, etiquette))
random.seed(42)
random.shuffle(combined)
images, labels = zip(*combined)
images = list(images)
labels = list(labels)


In [ ]:
import numpy as np
X = np.array(images)
y = np.array(labels)
# print(labels)
elements, counts = np.unique(y, return_counts=True)
for element, count in zip(elements, counts):
    print(f"{element}: {count}")

In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(x=y, order=classes)

plt.title("Repartition des images par classe")
plt.xlabel("Classe")
plt.ylabel("Nombre d'images")
plt.xticks(rotation=15)
plt.show()

In [ ]:
#  AUTOTUNE = tf.data.AUTOTUNE
# dataset = tf.data.Dataset.from_tensor_slices((images, etiquette))
# dataset = dataset.map()

In [ ]:

classes = sorted(np.unique(y))
for class_name in classes:
    # print(np.where(y == class_name)[0])
    index_list_imges = np.where(y == class_name)[0]
    print(index_list_imges[0])
    plt.imshow(X[index_list_imges[0]])
    plt.title(class_name)
    plt.axis("off")
    plt.show()

In [ ]:
total = len(y)
print(total)

for classe, count in zip(elements, counts):
    percentage = count/total*100
    print(f"{classe} has {count} images cad : {percentage.round(2)}%")
# df_classes = pd.DataFrame({
#     "Classe": classes,
#     "Nombre d'images": counts,
#     "Pourcentage": counts / len(y) * 100
# })
# df_classes

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()
y_encoded = le.fit_transform(y)

num_class = len(le.classes_)
# print(num_class)
y_categorical = to_categorical(
    y_encoded,
    num_classes = num_class
)
for i, class_name in enumerate(le.classes_):
    print(class_name, "=", i)
print(y_categorical.shape)
print(y_categorical)


In [ ]:
image_before = X[0].copy()
image_after = image_before.astype(np.float32) / 255.0
X = X.astype('float32') / 255.0

plt.imshow(image_before)
plt.title("image before")
plt.axis("off")
plt.show()

plt.imshow(image_after)
plt.title("image after")
plt.axis("off")
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_categorical,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)
print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test :", y_test.shape)


In [ ]:
from tensorflow.keras import layers

data_augmented = tf.keras.Sequential([
    layers.RandomRotation(0.05), 
    layers.RandomZoom(0.05), 
    layers.RandomTranslation(height_factor = 0.03, width_factor=0.03)])


augmented_sample = data_augmented(X_train[0], training=True)

plt.imshow(X_train[0])
plt.title("Image originale")
plt.axis("off")
plt.show()

plt.imshow(augmented_sample)
plt.title("Image augmentée")
plt.axis("off")
plt.show()




In [ ]:
batch_size = 30
Autotun = tf.data.AUTOTUNE
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(buffer_size=len(X_train))
train_ds = train_ds.map(lambda x,y: (data_augmented(x, training=True), y), num_parallel_calls=Autotun)
train_ds = train_ds.batch(batch_size).prefetch(Autotun)

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import time

from tensorflow.keras import layers, models
from tensorflow.keras.utils import plot_model

In [ ]:
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    layers.Conv2D(32, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(128, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D((2,2)),
    
    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(4, activation="softmax")
])
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
# plot_model(
#     model,
#     to_file="neurovision_cnn.png",
#     show_shapes=True,
#     show_layer_names=True,
#     show_trainable=True,
#     dpi=100
# )
# from IPython.display import display, Image
# display(Image(filename="neurovision_cnn.png"))

In [ ]:
epochs = 15
start_time = time.time()

history = model.fit(
    train_ds,
    epochs = epochs,
)
end_time = time.time()

training_duration = end_time - start_time
print(f"Durée totale : {training_duration / 60:.2f} minutes")

In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(history.history["accuracy"], label="Precision entrainement")
plt.plot(history.history["loss"], label="Loss entrainement")
plt.title("Evolution pendant l'entrainement")
plt.xlabel("Epoque")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=42,
)
print("Entrainement :", X_train_final.shape)
print("Validation :", X_val.shape)
print("Test :", X_test.shape)


def prepare_image(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label


train_ds = tf.data.Dataset.from_tensor_slices((X_train_final, y_train_final))
train_ds = train_ds.shuffle(len(X_train_final))
train_ds = train_ds.map(prepare_image, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.map(lambda image, label: (data_augmented(image, training=True),label),num_parallel_calls=AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.map(prepare_image, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_ds = test_ds.map(prepare_image,num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


In [ ]:

from pathlib import Path
from tensorflow.keras.callbacks import ModelCheckpoint, BackupAndRestore, EarlyStopping, ReduceLROnPlateau
Path("models").mkdir(parents=True, exist_ok=True)
Path("training_backup").mkdir(parents=True, exist_ok=True)

checkpoint = ModelCheckpoint(
    filepath="models/best_neurovision.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

backup = BackupAndRestore(
    backup_dir="training_backup",
    save_freq="epoch"
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    mode="min",
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    mode="min",
    verbose=1
)

callbacks = [
    checkpoint,
    backup,
    early_stopping,
    reduce_lr
]


In [ ]:
start_time = time.time()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)
end_time = time.time()
duration = end_time - start_time
print(f"Duree : {duration / 60:.2f} minutes")


In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Entrainement"
)
plt.plot(
    history.history["val_accuracy"],
    label="Validation"
)
plt.title("evolution de l'accuracy")
plt.xlabel("epoque")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()